In [1]:
!pip install -qU langchain-google-genai

## Configuración de Entorno y Conexión Segura con Google AI Studio

El primer paso fundamental en la evaluación y optimización de Modelos de Lenguaje Grande (LLMs) es establecer una arquitectura de conexión robusta, segura y escalable. Para esto utilizamos **Google AI Studio** como proveedor de los modelos fundacionales (familia Gemini) y **LangChain** como el orquestador.

### 1. Gestión de Credenciales

Exponer las API Keys en el código es un riesgo crítico. En entornos como Google Colab, utilizamos la herramienta `userdata` como bóveda de secretos. Luego, inyectamos ese secreto en el entorno virtual (`os.environ`) para que las librerías de LangChain lo absorban automáticamente sin que tengamos que pasarlo explícitamente como parámetro en cada función.

### 2. El Cliente LLM de LangChain

LangChain utiliza clases "Wrapper" (envoltorios). Al invocar `ChatGoogleGenerativeAI`, no estamos descargando el modelo a nuestra máquina; estamos instanciando un cliente que tomará nuestras instrucciones, las formateará según el estándar de Google, enviará la petición HTTP a la nube y nos devolverá un objeto estructurado con la respuesta. Para tareas de evaluación, es vital iniciar estas configuraciones con parámetros deterministas (como `temperature=0.0`) para que las métricas iniciales no se vean afectadas por la estocasticidad del modelo.

### Práctica: Configuración Segura e Invocación Base


In [4]:
import os
from typing import Optional

# Módulo de Colab para acceder a la bóveda de secretos de forma segura
from google.colab import userdata

# Cliente oficial de LangChain para la comunicación con Google AI Studio
from langchain_google_genai import ChatGoogleGenerativeAI


def inicializar_entorno_google() -> bool:
    """
    Extrae la clave API desde los secretos de Colab y la configura
    en las variables de entorno del sistema operativo virtual.
    """
    try:
        # Extraemos el valor usando el nombre exacto de la llave en Colab
        api_key_google = userdata.get('GOOGLE_API_KEY')

        # Inyectamos la llave en el entorno. LangChain está programado
        # para buscar esta variable específica antes de lanzar un error.
        os.environ["GOOGLE_API_KEY"] = api_key_google
        print("✅ Credenciales configuradas exitosamente.")
        return True

    except Exception as e:
        print(f"❌ Error al cargar credenciales: {e}")
        print("Asegúrate de configurar 'GOOGLE_API_KEY' en los secretos de Colab.")
        return False


def instanciar_modelo_evaluacion(nombre_modelo: str) -> Optional[ChatGoogleGenerativeAI]:
    """
    Crea una instancia del modelo Gemini optimizada para tareas de
    evaluación (determinista y sin aleatoriedad).
    """
    if not inicializar_entorno_google():
        return None

    try:
        # Instanciamos el cliente. Fijamos temperature en 0.0 para que el
        # modelo siempre elija la ruta probabilística más alta, asegurando
        # respuestas consistentes al evaluar.
        llm = ChatGoogleGenerativeAI(
            model=nombre_modelo,
            temperature=0.0
        )
        print(f"🤖 Conexión establecida con el modelo: {nombre_modelo}")
        return llm
    except Exception as e:
        print(f"❌ Error al instanciar el modelo: {e}")
        return None

# Probamos nuestra arquitectura base
llm_base = instanciar_modelo_evaluacion("gemini-3.5-flash")
if llm_base:
    # 'invoke' es el método síncrono estándar de LangChain para pedir una inferencia
    respuesta = llm_base.invoke("Confirma tu disponibilidad respondiendo 'Sistemas OK'.")
    print(f"Respuesta del modelo: {respuesta.content}")

✅ Credenciales configuradas exitosamente.
🤖 Conexión establecida con el modelo: gemini-3.5-flash
Respuesta del modelo: [{'type': 'text', 'text': 'Sistemas OK', 'extras': {'signature': 'EqkECqYEARFNMg81xcn3plrzXj2bztk2BpfSxRW1uUs0jAEJhAT095XhpMVT+2XjTo8a/3izTkkUV257kkjn/dSqncqxSeca4KitzAVG2yREFj8NFPtvKGY+5r1wTwrg/4RJhsaiqh0isNy++/UdKZ/on/SPxQ+fziw9f/fJmViKj0S4LNMxi+0rAJjXuXS3eBJHBxnVWcpKUvbyeeO21poTbKprCW84PMI1y2ZTWtDybLsTcXa+WI4MM3i7FXf5eQ1ZXBPA+M39Y6NOQoxKV6K4g59gRSbee2eU7wsMH1FzZVssh4BIQYhasKSwPIEkI1zdEeSes0J+QY+YQhdjwJV43jieunjYdYNYq/sksnC3rDmF8gy9iT7xH4Xw3TYzSkwgg4i5O3SOIo0WL7oiRH1X8joV8J+P6w7YZNmgKJ3V2ifFRD3XejEeoRdmisbIM+v7NYlnQ/JTl5Qe1kp/HFqIFg2jgXVPN2HIIUELnQLb+RJ9TapODXGtZiFGhZHxlBpJMUV/sVTO688ueVydMiNAV6K1dDzU7J9RVryy4bfrhPUyBdKbBUzw22fG1ctsVcbl7ZZRrN+GgOzTkPmDCs2kB70ShKN3iaYgS+DLHtVi8ubwuo/kBgyLaJdYpq6cdkMVdMJeM1qPHQbW7WvkumHDKu7um7DfkFKZwVx3jY5BCMMoW+TEZJnjuskXnjFtrxH/+TA81Hq5vSnfJCpgvyVzut7bVt9mfs0VEQ=='}}]


### Ejercicio Práctico: Validador de Disponibilidad Multi-Modelo

**Objetivo:** Desarrollar un script que, dado un listado de modelos disponibles en Google AI Studio, intente conectar con cada uno, envíe un prompt simple, y devuelva un reporte estructurado de cuáles están disponibles y responden correctamente.


In [24]:
import os
from typing import Any
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

# Carga de credenciales
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


def obtener_texto(respuesta: Any) -> str:
    """Función de extracción segura para evitar AttributeError."""
    contenido = respuesta.content
    if isinstance(contenido, str):
        return contenido.strip()
    elif isinstance(contenido, list) and len(contenido) > 0:
        primer_bloque = contenido[0]
        if isinstance(primer_bloque, dict) and "text" in primer_bloque:
            return primer_bloque["text"].strip()
        return str(primer_bloque).strip()
    return str(contenido).strip()

def evaluar_disponibilidad_modelos(lista_modelos: list[str]) -> dict[str, str]:
    """
    Itera sobre una lista de nombres de modelos, ejecuta una consulta de prueba
    y registra el resultado o el tipo de error capturado.
    """
    reporte = {}
    prompt_prueba = "Responde únicamente con la palabra 'Activo'."

    print("\n--- INICIANDO DIAGNÓSTICO DE MODELOS ---")

    for modelo in lista_modelos:
        print(f"⏳ Evaluando conexión con: {modelo}...")
        try:
            # Instanciamos con max_retries=1 para no demorar la prueba si el modelo falla
            llm = ChatGoogleGenerativeAI(
                model=modelo,
                temperature=0.0,
                max_retries=1
            )

            # Ejecutamos la llamada
            respuesta_raw = llm.invoke(prompt_prueba)

            # Extraemos el texto de forma segura
            texto_respuesta = obtener_texto(respuesta_raw)
            reporte[modelo] = f"✅ ÉXITO -> Respuesta: '{texto_respuesta}'"

        except Exception as e:
            # Capturamos el tipo exacto de excepción para el reporte
            tipo_error = type(e).__name__
            reporte[modelo] = f"❌ FALLÓ -> [{tipo_error}]: {e}"

    return reporte


# =========================================
# EJECUCIÓN DEL EJERCICIO Y SALIDA
# =========================================
# Probamos el modelo que funciona junto con una cadena inexistente para validar el manejo de errores
modelos_a_probar = [
    "gemini-3.6-flash",
    "modelo-inexistente-para-prueba"
]

resultados = evaluar_disponibilidad_modelos(modelos_a_probar)

print("\n" + "="*60)
print("📊 REPORTE FINAL DE CONECTIVIDAD")
print("="*60)
for nombre_modelo, estado in resultados.items():
    print(f"• {nombre_modelo}: {estado}")


--- INICIANDO DIAGNÓSTICO DE MODELOS ---
⏳ Evaluando conexión con: gemini-3.6-flash...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


⏳ Evaluando conexión con: modelo-inexistente-para-prueba...

📊 REPORTE FINAL DE CONECTIVIDAD
• gemini-3.6-flash: ✅ ÉXITO -> Respuesta: 'Activo'
• modelo-inexistente-para-prueba: ❌ FALLÓ -> [GoogleModelNotFoundError]: Error calling model 'modelo-inexistente-para-prueba' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/modelo-inexistente-para-prueba is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}


# Control de Hiperparámetros (Temperatura, Top-P, Top-K y Límite de Tokens)

## Gobernanza Probabilística de las Generaciones

Los Modelos de Lenguaje Grande seleccionan iterativamente el token (subpalabra) más probable a partir de una distribución de probabilidad sobre su vocabulario.

Para controlar la creatividad o el determinismo del modelo sin alterar la instrucción base, ajustamos cuatro **hiperparámetros de muestreo**:

1. **Temperatura (****`temperature`****):**

   * **`0.0`** **(Determinista / Greedy):** Selecciona siempre el token con mayor probabilidad. Es el valor indispensable para tareas analíticas, auditorías, código y extracción de datos.
   * **`1.0`** **o superior (Creativo):** Aplana la curva de probabilidad, permitiendo términos menos frecuentes para lluvia de ideas.
2. **Top-P (****`top_p`****):** Muestreo por probabilidad acumulada. Si `top_p = 0.90`, retiene únicamente los tokens que acumulan el 90% superior de probabilidad y descarta el resto.
3. **Top-K (****`top_k`****):** Limita la selección a los $K$ tokens candidatos con mayor probabilidad.
4. **Límite de Tokens (****`max_output_tokens`****):** Restringe la longitud máxima de la respuesta para controlar costos y prevenir bucles.


In [12]:
import os
from typing import Any
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

# Carga de credenciales
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


def obtener_texto(respuesta: Any) -> str:
    """
    Extractor universal de texto.
    Soporta cadenas, diccionarios, objetos TextBlock y listas de fragmentos.
    """
    contenido = getattr(respuesta, "content", respuesta)

    # Caso 1: Cadena de texto simple
    if isinstance(contenido, str):
        return contenido.strip()

    # Caso 2: Lista de fragmentos / bloques
    if isinstance(contenido, list):
        partes = []
        for elem in contenido:
            if isinstance(elem, str):
                partes.append(elem)
            elif isinstance(elem, dict):
                # Extrae de diccionarios {'text': '...'} o {'content': '...'}
                val = elem.get("text") or elem.get("content") or ""
                partes.append(str(val))
            else:
                # Extrae de objetos que tienen atributo .text o .content
                val = getattr(elem, "text", None) or getattr(elem, "content", None) or str(elem)
                partes.append(str(val))

        texto_unido = "".join(partes).strip()
        if texto_unido:
            return texto_unido

    # Caso 3: Objeto directo con atributo .text
    if hasattr(respuesta, "text") and not callable(getattr(respuesta, "text")):
        return str(respuesta.text).strip()

    return str(contenido).strip()


# -------------------------------------------------------------------------
# 1. Configuración Analítica Determinista (Temperature = 0.0)
# -------------------------------------------------------------------------
llm_analitico = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.0
)

# -------------------------------------------------------------------------
# 2. Configuración Creativa (Temperature = 1.0)
# -------------------------------------------------------------------------
llm_creativo = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=1.0
)

prompt_evaluacion = "Resume en una sola oración la importancia del balance general."

print("--- DEMOSTRACIÓN DE CONFIGURACIONES ---")
res_analitica = llm_analitico.invoke(prompt_evaluacion)
res_creativa = llm_creativo.invoke(prompt_evaluacion)

print(f"📊 ANALÍTICA (Temp 0.0):\n{obtener_texto(res_analitica)}\n")
print(f"🎨 CREATIVA (Temp 1.0):\n{obtener_texto(res_creativa)}\n")

--- DEMOSTRACIÓN DE CONFIGURACIONES ---
📊 ANALÍTICA (Temp 0.0):
El balance general es fundamental porque ofrece una radiografía clara de la salud financiera de una empresa en un momento determinado, permitiendo evaluar su solvencia, liquidez y capacidad para tomar decisiones estratégicas.

🎨 CREATIVA (Temp 1.0):
El balance general es fundamental porque ofrece una radiografía clara de la salud financiera de una empresa en un momento determinado, permitiendo evaluar su liquidez, solvencia y capacidad para tomar decisiones estratégicas y de inversión.



### Ejercicio Práctico: Medidor de Consistencia de Respuestas

Ejecuta este bloque para comprobar cómo varía la respuesta a temperatura `0.0` frente a temperatura `1.2` a lo largo de 3 repeticiones:


EJEMPLO DE SALIDA :
⏳ Evaluando 3 ejecuciones con Temperatura = 0.0...
⏳ Evaluando 3 ejecuciones con Temperatura = 1.2...

============================================================
📊 RESULTADOS DEL EXPERIMENTO DE CONSISTENCIA
============================================================

--- TEMPERATURA: 0.0 ---
  [Intento 1]: Un activo no corriente es un bien o derecho propiedad de una empresa que no está destinado para la venta y cuya permanencia, uso o conversión en efectivo se prevé a largo plazo, superando el período de un año.
  [Intento 2]: Un activo no corriente es un bien o derecho propiedad de una empresa que no está destinado para la venta y cuya permanencia, uso o conversión en efectivo se prevé a largo plazo, superando el período de un año.
  [Intento 3]: Un activo no corriente es un bien o derecho de una empresa que está destinado a permanecer en su patrimonio por un periodo superior a un año, ya que no se adquiere para su venta inmediata sino para asegurar el funcionamiento de la actividad productiva a largo plazo.

--- TEMPERATURA: 1.2 ---
  [Intento 1]: Un activo no corriente es un recurso económico propiedad de una empresa que no está destinado para la venta a corto plazo y cuya recuperación, consumo o conversión en efectivo se prevé en un periodo superior a un año.
  [Intento 2]: Un activo no corriente es aquel bien o derecho que posee una empresa y que no está destinado a convertirse en dinero líquido a corto plazo, sino a permanecer en el patrimonio por un periodo superior a un año para asegurar el funcionamiento del negocio.
  [Intento 3]: Un activo no corriente es aquel bien o derecho propiedad de una empresa que está destinado a p

In [26]:
def experimento_consistencia(
    prompt: str,
    temperaturas: list[float],
    repeticiones: int = 3
) -> dict[float, list[str]]:
    """
    Ejecuta un prompt repetidamente variando la temperatura para comprobar
    la estabilidad de la respuesta completa del modelo.
    """
    reporte = {}

    for temp in temperaturas:
        print(f"⏳ Evaluando {repeticiones} ejecuciones con Temperatura = {temp}...")
        reporte[temp] = []

        llm = ChatGoogleGenerativeAI(
            model="gemini-3.6-flash",
            temperature=temp
        )

        for _ in range(repeticiones):
            respuesta = llm.invoke(prompt)
            texto_completo = obtener_texto(respuesta)
            reporte[temp].append(texto_completo)

    return reporte


# =========================================
# EJECUCIÓN DEL EXPERIMENTO
# =========================================
consulta = "Define qué es un activo no corriente en exactamente una oración."
lista_temp = [0.0, 1.2]

resultados = experimento_consistencia(
    prompt=consulta,
    temperaturas=lista_temp,
    repeticiones=3
)

print("\n" + "="*60)
print("📊 RESULTADOS DEL EXPERIMENTO DE CONSISTENCIA")
print("="*60)

for temp, respuestas in resultados.items():
    print(f"\n--- TEMPERATURA: {temp} ---")
    for idx, resp in enumerate(respuestas, 1):
        print(f"  [Intento {idx}]: {resp}")

⏳ Evaluando 3 ejecuciones con Temperatura = 0.0...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


⏳ Evaluando 3 ejecuciones con Temperatura = 1.2...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



📊 RESULTADOS DEL EXPERIMENTO DE CONSISTENCIA

--- TEMPERATURA: 0.0 ---
  [Intento 1]: Un activo no corriente es un bien o derecho propiedad de una empresa que está destinado a permanecer en su patrimonio por un periodo superior a un año para apoyar sus operaciones y que no se puede convertir fácilmente en efectivo a corto plazo.
  [Intento 2]: Un activo no corriente es un bien o derecho propiedad de una empresa que no está destinado a convertirse en efectivo en el corto plazo, sino a permanecer en la organización durante más de un año para respaldar sus operaciones a largo plazo.
  [Intento 3]: Un activo no corriente es un bien o recurso económico de una empresa que está destinado a permanecer en su patrimonio por un periodo superior a un año y que no se pretende convertir en efectivo en el corto plazo.

--- TEMPERATURA: 1.2 ---
  [Intento 1]: Un activo no corriente es un recurso económico de una empresa que no está destinado a convertirse en efectivo a corto plazo, sino que se mantie

# Optimización de Prompts mediante `ChatPromptTemplate` y Técnicas Few-Shot

## Estructuración Contextual del Aprendizaje en Ventana

Escribir prompts como cadenas de texto planas (*f-strings*) introduce fragilidad en los sistemas de producción. Para construir arquitecturas mantenibles, LangChain abstrae la construcción de mensajes mediante **`ChatPromptTemplate`**, lo que permite separar la lógica de las instrucciones, los datos variables y los roles de la conversación.

### Roles en Modelos Conversacionales

Gemini y LangChain estructuran la interacción utilizando tres roles fundamentales:

* **System (****`system`****):** Define las reglas globales, restricciones de comportamiento, tono y límites operativos. Actúa como la instrucción maestra.
* **Human (****`human`****):** Representa la entrada variable del usuario final o la consulta enviada en tiempo de ejecución.
* **AI (****`ai`****):** Simula respuestas previas del modelo o ejemplos de salida deseada.

### Aprendizaje en Contexto (Few-Shot Prompting)

Cuando una instrucción es compleja, la técnica más efectiva para alinear al LLM no es escribir explicaciones más largas, sino **proveer ejemplos de entrada/salida**. Mediante `FewShotChatMessagePromptTemplate`, insertamos pares de demostración directamente en el prompt. El modelo infiere el patrón probabilístico exacto y reproduce el formato con alta fidelidad sin requerir reentrenamiento (*fine-tuning*).

### Práctica: Clasificación Contable con Few-Shot Prompting y LCEL

El siguiente código utiliza `ChatPromptTemplate` junto con la función `obtener_texto` para garantizar la recepción completa del resultado.


In [15]:
import os
from typing import Any
from google.colab import userdata

# Importación de clases de plantillas de LangChain
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate
)
from langchain_google_genai import ChatGoogleGenerativeAI

# Carga de credenciales desde los secretos de Colab
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


def obtener_texto(respuesta: Any) -> str:
    """Extractor universal de texto seguro para respuestas de LangChain."""
    contenido = getattr(respuesta, "content", respuesta)
    if isinstance(contenido, str):
        return contenido.strip()
    if isinstance(contenido, list):
        partes = []
        for elem in contenido:
            if isinstance(elem, str):
                partes.append(elem)
            elif isinstance(elem, dict):
                partes.append(str(elem.get("text") or elem.get("content") or ""))
            else:
                partes.append(getattr(elem, "text", None) or str(elem))
        return "".join(partes).strip()
    return str(contenido).strip()


# 1. Definimos la lista de ejemplos Few-Shot (Pares entrada/salida)
ejemplos_auditoria = [
    {
        "transaccion": "Pago de $450 por servicio de internet corporativo",
        "clasificacion": "CATEGORÍA: Gasto Operativo | CUENTA: Servicios Básicos | DEDUCIBLE: Sí"
    },
    {
        "transaccion": "Compra de terreno industrial por $120,000",
        "clasificacion": "CATEGORÍA: Activo No Corriente | CUENTA: Propiedad y Terrenos | DEDUCIBLE: No"
    }
]

# 2. Diseñamos la plantilla individual para los ejemplos
plantilla_ejemplo = ChatPromptTemplate.from_messages([
    ("human", "{transaccion}"),
    ("ai", "{clasificacion}")
])

# 3. Empaquetamos los ejemplos dentro del componente Few-Shot
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=plantilla_ejemplo,
    examples=ejemplos_auditoria
)

# 4. Construimos la plantilla final uniendo Sistema + Ejemplos + Entrada del Usuario
prompt_sistema = (
    "Eres un auditor contable senior. Clasifica las transacciones recibidas "
    "siguiendo estrictamente la estructura mostrada en los ejemplos."
)

prompt_final = ChatPromptTemplate.from_messages([
    ("system", prompt_sistema),
    few_shot_prompt,
    ("human", "{transaccion}")
])

# 5. Instanciamos el modelo con temperatura 0.0 para alineación estricta
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.0
)

# 6. Unimos Prompt y LLM mediante LCEL (LangChain Expression Language)
cadena_clasificadora = prompt_final | llm

# Ejecución de prueba
consulta_nueva = {"transaccion": "Adquisición de 5 licencias de software contable por $1,500"}
resultado_raw = cadena_clasificadora.invoke(consulta_nueva)

print("--- RESULTADO DE CLASIFICACIÓN FEW-SHOT ---")
print(obtener_texto(resultado_raw))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- RESULTADO DE CLASIFICACIÓN FEW-SHOT ---
CATEGORÍA: Activo No Corriente | CUENTA: Activos Intangibles | DEDUCIBLE: Sí


### Ejercicio Práctico: Extractor de Divisas y Montos Financieros

**Objetivo:** Crear una cadena ejecutable con `ChatPromptTemplate` y `FewShotChatMessagePromptTemplate` que reciba párrafos en texto libre y extraiga únicamente los montos y sus respectivas divisas en un formato estandarizado.


EJEMPLO: -- TEXTO ENTRADA ---
El cliente abonó 2400 USD en la mañana. Por la tarde, la sucursal de Santiago registró un ingreso adicional de 850000 CLP y pagó un proveedor con 300 EUR.

--- SALIDA EXTRAÍDA POR EL MODELO ---
USD: 2400.00 | CLP: 850000.00 | EUR: 300.00

In [28]:
import os
from typing import Any
from google.colab import userdata
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate
)
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


def obtener_texto(respuesta: Any) -> str:
    """Extractor universal de texto."""
    contenido = getattr(respuesta, "content", respuesta)
    if isinstance(contenido, str):
        return contenido.strip()
    if isinstance(contenido, list):
        partes = []
        for elem in contenido:
            if isinstance(elem, str):
                partes.append(elem)
            elif isinstance(elem, dict):
                partes.append(str(elem.get("text") or elem.get("content") or ""))
            else:
                partes.append(getattr(elem, "text", None) or str(elem))
        return "".join(partes).strip()
    return str(contenido).strip()


def crear_cadena_extractor_divisas():
    """
    Construye y retorna una cadena LCEL entrenada con Few-Shot
    para extracción estandarizada de valores monetarios.
    """
    # Lista de ejemplos de entrenamiento en contexto
    ejemplos_extraccion = [
        {
            "texto_libre": "Ayer gastamos 350 USD en transporte y luego cobramos 1200 EUR por la consultoría.",
            "extraccion": "USD: 350.00 | EUR: 1200.00"
        },
        {
            "texto_libre": "Se transfirieron 50000 CLP para caja chica y se pagaron 150 USD de servidor.",
            "extraccion": "CLP: 50000.00 | USD: 150.00"
        }
    ]

    # Plantilla de formato para cada ejemplo
    plantilla_ejemplo = ChatPromptTemplate.from_messages([
        ("human", "{texto_libre}"),
        ("ai", "{extraccion}")
    ])

    # Ensamblado del bloque Few-Shot
    few_shot_block = FewShotChatMessagePromptTemplate(
        example_prompt=plantilla_ejemplo,
        examples=ejemplos_extraccion
    )

    # Prompt global de la cadena
    prompt_completo = ChatPromptTemplate.from_messages([
        (
            "system",
            "Extrae todos los montos y divisas presentes en el texto. "
            "Responde ÚNICAMENTE con la estructura 'DIVISA: MONTO' separada por tubos (|). "
            "No agregues explicaciones adicionales."
        ),
        few_shot_block,
        ("human", "{texto_libre}")
    ])

    # Modelo configurado de forma determinista
    llm = ChatGoogleGenerativeAI(
        model="gemini-3.6-flash",
        temperature=0.0
    )

    # Creación de la cadena LCEL (Prompt -> LLM)
    return prompt_completo | llm


# =========================================
# EJECUCIÓN Y SALIDA DEL EJERCICIO
# =========================================
cadena_extractor = crear_cadena_extractor_divisas()

parrafo_prueba = (
    "El cliente abonó 2400 USD en la mañana. Por la tarde, la sucursal de Santiago "
    "registró un ingreso adicional de 850000 CLP y pagó un proveedor con 300 EUR."
)

print("\n--- TEXTO ENTRADA ---")
print(parrafo_prueba)

resultado_extraccion = cadena_extractor.invoke({"texto_libre": parrafo_prueba})

print("\n--- SALIDA EXTRAÍDA POR EL MODELO ---")
print(obtener_texto(resultado_extraccion))


--- TEXTO ENTRADA ---
El cliente abonó 2400 USD en la mañana. Por la tarde, la sucursal de Santiago registró un ingreso adicional de 850000 CLP y pagó un proveedor con 300 EUR.


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



--- SALIDA EXTRAÍDA POR EL MODELO ---
USD: 2400.00 | CLP: 850000.00 | EUR: 300.00


# &#x20;Salidas Estructuradas con Parsers de LangChain y Pydantic para Validación

## De Texto Plano a Contratos de Datos Rígidos

En entornos de producción, confiar en que un LLM devuelva siempre texto en el formato correcto (como un JSON perfectamente formateado) usando solo instrucciones en el prompt es riesgoso. Un simple fallo sintáctico, una comilla faltante o una clave con nombre diferente romperá las integraciones con bases de datos, APIs o procesos posteriores.

Para resolver esto de forma determinista, LangChain se integra con **Pydantic**, la librería estándar de Python para validación de datos mediante esquemas de tipos estrictos.

### Componentes Clave:

1. **Esquema Pydantic (****`BaseModel`****):** Define los campos, tipos de datos (`int`, `float`, `bool`, `list`, `str`) y descripciones detalladas de cada propiedad.
2. **`PydanticOutputParser`****:** Genera automáticamente un conjunto de instrucciones de formato JSON (`format_instructions`) que se inyectan en el prompt del sistema y se encarga de convertir la respuesta cruda del LLM en una instancia tipada de Python.
3. **Manejo y Validación de Tipos:** Si el modelo genera un valor inválido (por ejemplo, un texto donde se esperaba un número flotante), Pydantic lanza una excepción controlada en lugar de propagar un dato corrupto al sistema.

### Práctica: Auditoría Contable con Esquema Pydantic

En este ejemplo, definimos una estructura estricta para la clasificación de transacciones financieras y la procesamos mediante una cadena LCEL (`Prompt | LLM | Parser`).


In [17]:
import os
from typing import Any
from google.colab import userdata
from pydantic import BaseModel, Field

# Componentes de LangChain para parsing y prompts
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

# Carga de credenciales desde los secretos de Colab
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


# 1. Definición del Esquema Pydantic (El contrato de datos)
class TransaccionAuditada(BaseModel):
    monto: float = Field(
        description="Monto numérico exacto de la transacción (sin símbolos de moneda)"
    )
    divisa: str = Field(
        description="Código ISO de 3 letras de la moneda (ej. USD, EUR, CLP)"
    )
    categoria: str = Field(
        description="Categoría contable (ej. 'Servicios', 'Activo Fijo', 'Nómina', 'Suministros')"
    )
    es_deducible: bool = Field(
        description="True si el gasto cumple criterios de deducibilidad fiscal, False en caso contrario"
    )
    justificacion_criterio: str = Field(
        description="Explicación técnica de 1 oración que justifica la categoría y deducibilidad"
    )


# 2. Creación del Parser a partir del Esquema
parser_transaccion = PydanticOutputParser(pydantic_object=TransaccionAuditada)

# 3. Diseño del Prompt incorporando las instrucciones automáticas de formato
prompt_sistema = (
    "Eres un sistema de auditoría contable automatizado. "
    "Analiza la transacción recibida y extrae la información requerida.\n"
    "Debes responder ÚNICAMENTE con la estructura JSON especificada a continuación.\n\n"
    "{format_instructions}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", prompt_sistema),
    ("human", "Transacción a procesar: {descripcion_transaccion}")
])

# Inyectamos de forma parcial las instrucciones de formato generadas por Pydantic
prompt_configurado = prompt.partial(
    format_instructions=parser_transaccion.get_format_instructions()
)

# 4. Instanciación del Modelo (Temperatura 0.0 para adherencia estricta al esquema)
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.0
)

# 5. Ensamblado de la Cadena LCEL con Parsing Automático
cadena_auditara = prompt_configurado | llm | parser_transaccion

# Ejecución de prueba
entrada = "Se pagó la factura N° 4021 por $3,450.00 USD correspondiente al servidor Cloud de AWS del mes."
objeto_resultado = cadena_auditara.invoke({"descripcion_transaccion": entrada})

print("--- OBJETO PYDANTIC PARSEADO Y TIPADO ---")
print(f"Tipo de Objeto: {type(objeto_resultado)}")
print(f"Monto: {objeto_resultado.monto} ({type(objeto_resultado.monto)})")
print(f"Divisa: {objeto_resultado.divisa}")
print(f"Categoría: {objeto_resultado.categoria}")
print(f"¿Es Deducible?: {objeto_resultado.es_deducible} ({type(objeto_resultado.es_deducible)})")
print(f"Justificación: {objeto_resultado.justificacion_criterio}")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- OBJETO PYDANTIC PARSEADO Y TIPADO ---
Tipo de Objeto: <class '__main__.TransaccionAuditada'>
Monto: 3450.0 (<class 'float'>)
Divisa: USD
Categoría: Servicios
¿Es Deducible?: True (<class 'bool'>)
Justificación: El gasto corresponde a un servicio tecnológico de hosting en la nube estrictamente indispensable para la operación del negocio, por lo que resulta totalmente deducible de impuestos.


### Ejercicio Práctico: Evaluación de Riesgo de Facturas

**Objetivo:** Construir un sistema de evaluación de riesgo contable que procese el texto de una factura y devuelva un objeto estructurado `ReporteRiesgoFactura` con alertas, puntuación numérica de riesgo e instrucciones de seguimiento.


In [31]:
import os
from typing import List, Literal
from google.colab import userdata
from pydantic import BaseModel, Field

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


# 1. Definición del Esquema de Riesgo
class ReporteRiesgoFactura(BaseModel):
    id_factura: str = Field(
        description="Identificador o número de folio de la factura analizada"
    )
    nivel_riesgo: Literal["BAJO", "MEDIO", "ALTO"] = Field(
        description="Nivel de riesgo asignado a la transacción según coherencia y montos"
    )
    puntuacion_riesgo: int = Field(
        description="Puntaje de riesgo de 0 (Sin riesgo) a 100 (Riesgo crítico)"
    )
    banderas_rojas: List[str] = Field(
        description="Lista de anomalias o inconsistencias detectadas (vacío si no hay)"
    )
    accion_recomendada: str = Field(
        description="Instrucción concreta para el departamento de pagos"
    )


def crear_evaluador_riesgo():
    """
    Crea y retorna una cadena LCEL que garantiza una salida parseada
    según el esquema ReporteRiesgoFactura.
    """
    parser = PydanticOutputParser(pydantic_object=ReporteRiesgoFactura)

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "Eres un agente especialista en prevención de fraude contable.\n"
            "Analiza el detalle de la factura enviada y evalúa su nivel de riesgo.\n"
            "Cumple estrictamente con el siguiente esquema de salida:\n{format_instructions}"
        ),
        ("human", "Detalle de la factura: {detalle_factura}")
    ]).partial(format_instructions=parser.get_format_instructions())

    llm = ChatGoogleGenerativeAI(
        model="gemini-3.6-flash",
        temperature=0.0
    )

    return prompt | llm | parser


# =========================================
# EJECUCIÓN COMPLETA DEL EJERCICIO
# =========================================
evaluador_riesgo = crear_evaluador_riesgo()

factura_sospechosa = (
    "Factura borrosa N° F-9901 recibida a nombre de 'Consultorías Generales S.A.' "
    "por un monto de $48,000.00 USD. Concepto: 'Servicios varios de asesoría e imprevistos'. "
    "No se adjunta contrato de respaldo y la cuenta bancaria de destino pertenece a un país distinto."
)

print("\n--- PROCESANDO FACTURA PARA EVALUACIÓN DE RIESGO ---")
reporte: ReporteRiesgoFactura = evaluador_riesgo.invoke({"detalle_factura": factura_sospechosa})

print("\n============================================================")
print("📄 REPORTE DE EVALUACIÓN DE RIESGO (ESTRUCTURADO)")
print("============================================================")
print(f"• ID Factura           : {reporte.id_factura}")
print(f"• Nivel de Riesgo      : {reporte.nivel_riesgo}")
print(f"• Puntuación de Riesgo : {reporte.puntuacion_riesgo} / 100")
print(f"• Banderas Rojas       :")
for bandera in reporte.banderas_rojas:
    print(f"   ⚠️ {bandera}")
print(f"• Acción Recomendada   : {reporte.accion_recomendada}")


--- PROCESANDO FACTURA PARA EVALUACIÓN DE RIESGO ---


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



📄 REPORTE DE EVALUACIÓN DE RIESGO (ESTRUCTURADO)
• ID Factura           : F-9901
• Nivel de Riesgo      : ALTO
• Puntuación de Riesgo : 92 / 100
• Banderas Rojas       :
   ⚠️ Documento borroso o con visibilidad deficiente
   ⚠️ Concepto de facturación vago e impreciso ('Servicios varios de asesoría e imprevistos')
   ⚠️ Ausencia de contrato de respaldo para un monto elevado ($48,000.00 USD)
   ⚠️ Cuenta bancaria de destino ubicada en un país distinto al del proveedor
• Acción Recomendada   : Bloquear o detener el pago de inmediato. Solicitar una copia legible del documento, requerir el contrato de prestación de servicios debidamente firmado junto con los entregables que justifiquen el servicio, y validar la titularidad y justificación de la cuenta bancaria en el extranjero.


# Técnicas de Inyección de Contexto y Mitigación de Alucinaciones

## Anclaje en Contexto (*Grounding*) y Control de Veracidad

Las alucinaciones en los LLM suceden porque los modelos priorizan la **coherencia probabilística** sobre la **verdad fáctica**. Cuando el modelo no conoce un dato específico o no lo tiene en su ventana de contexto, su mecanismo de muestreo intentará completar la secuencia con tokens plausiblemente correctos pero potencialmente falsos.

En entornos empresariales y financieros, la forma más efectiva de eliminar las alucinaciones no es reentrenar el modelo, sino aplicar **Inyección de Contexto Anclado (*****In-Context Grounding*****)**.

### Estrategias Anti-Alucinación en LangChain:

1. **Fronteras de Conocimiento Rígidas:** Instruir explícitamente al modelo en el `SystemMessage` para que opere en un entorno de "mundo cerrado", limitando su universo de conocimiento únicamente al texto provisto en el prompt.
2. **Protocolo de Salida por Incumplimiento:** Definir una respuesta estándar obligatoria (ej. *"INFORMACIÓN NO ENCONTRADA EN EL CONTEXTO"*) cuando la consulta no pueda ser respondida con los datos proporcionados.
3. **Exigencia de Citas Textuales (*****Verifiability*****):** Obligar al modelo a extraer fragmentos textuales exactos (*quotes*) que respalden cada afirmación realizada.

### Práctica: Sistema de Preguntas y Respuestas Anclado a Documento

El siguiente script inyecta una política interna de reembolsos y restringe la libertad del modelo mediante instrucciones estrictas de anclaje.


In [19]:
import os
from typing import Any
from google.colab import userdata
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


def obtener_texto(respuesta: Any) -> str:
    """Extractor universal de texto seguro para respuestas de LangChain."""
    contenido = getattr(respuesta, "content", respuesta)
    if isinstance(contenido, str):
        return contenido.strip()
    if isinstance(contenido, list):
        partes = []
        for elem in contenido:
            if isinstance(elem, str):
                partes.append(elem)
            elif isinstance(elem, dict):
                partes.append(str(elem.get("text") or elem.get("content") or ""))
            else:
                partes.append(getattr(elem, "text", None) or str(elem))
        return "".join(partes).strip()
    return str(contenido).strip()


# 1. Documento de Contexto Anclado (Fuente única de verdad)
politica_viajes_2026 = """
POLÍTICA INTERNA DE REEMBOLSOS Y VIAJES 2026 (CÓDIGO: POL-FIN-02)
1. Alojamientos: El límite máximo permitido para hotel es de $180 USD por noche en ciudades principales y $120 USD en secundarias.
2. Alimentación: Se asigna un viático diario de $60 USD sin necesidad de presentar comprobantes de consumo individual.
3. Vuelos: Todo vuelo internacional debe ser reservado con al menos 14 días de anticipación en clase económica. No se autoriza business class salvo vuelos superiores a 10 horas contiguas.
4. Taxis y Uber: Permitidos únicamente para trayectos Aeropuerto-Hotel y Hotel-Oficina.
"""

# 2. Diseñamos el Prompt con Instrucción Restrictiva (Mundo Cerrado)
prompt_anclado = ChatPromptTemplate.from_messages([
    (
        "system",
        "Eres un auditor estricto de cumplimiento normativo. "
        "Responde la pregunta del usuario utilizando ÚNICAMENTE la información provista en el 'CONTEXTO DE REFERENCIA'.\n"
        "REGLAS OBLIGATORIAS:\n"
        "- Si la respuesta no se encuentra explícitamente escrita en el contexto, debes responder exactamente: "
        "'INFORMACIÓN NO DISPONIBLE EN EL CONTEXTO'.\n"
        "- No uses conocimientos previos ni asunciones externas bajo ninguna circunstancia.\n\n"
        "CONTEXTO DE REFERENCIA:\n{contexto}"
    ),
    ("human", "Pregunta: {pregunta}")
])

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.0
)

cadena_anclada = prompt_anclado | llm

# --- PRUEBA 1: Pregunta presente en el contexto ---
p1 = "Solo volaré 6 horas a Bogotá, ¿puedo viajar en Business Class?"
res1 = cadena_anclada.invoke({"contexto": politica_viajes_2026, "pregunta": p1})

print("--- PRUEBA 1 (DATO PRESENTE) ---")
print(f"Pregunta: {p1}")
print(f"Respuesta del LLM:\n{obtener_texto(res1)}\n")

# --- PRUEBA 2: Pregunta ausente del contexto (Intento de alucinación) ---
p2 = "¿Cuál es el monto máximo permitido para cenas de representación con clientes?"
res2 = cadena_anclada.invoke({"contexto": politica_viajes_2026, "pregunta": p2})

print("--- PRUEBA 2 (DATO AUSENTE / PREVENCIÓN DE ALUCINACIÓN) ---")
print(f"Pregunta: {p2}")
print(f"Respuesta del LLM:\n{obtener_texto(res2)}")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- PRUEBA 1 (DATO PRESENTE) ---
Pregunta: Solo volaré 6 horas a Bogotá, ¿puedo viajar en Business Class?
Respuesta del LLM:
No. Según la política interna (sección 3), no se autoriza el viaje en *business class* salvo para vuelos superiores a 10 horas contiguas, por lo que para un vuelo de 6 horas debe viajar en clase económica.



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- PRUEBA 2 (DATO AUSENTE / PREVENCIÓN DE ALUCINACIÓN) ---
Pregunta: ¿Cuál es el monto máximo permitido para cenas de representación con clientes?
Respuesta del LLM:
INFORMACIÓN NO DISPONIBLE EN EL CONTEXTO


### Ejercicio Práctico: Verificador de Aseveraciones con Cita Textual

**Objetivo:** Desarrollar una cadena de verificación fáctica (*Fact-Checker*) que reciba una afirmación y un fragmento de informe auditado. La cadena debe evaluar si la afirmación es `SOPORTADO`, `CONTRADICTORIO` o `NO_MENCIONADO`, retornando adicionalmente la cita textual exacta del documento original como evidencia.


In [ ]:
documento_auditoria = """
INFORME DE AUDITORÍA ANUAL 2025 - GRUPO FINANCIERO SUR
En el cuarto trimestre de 2025, los ingresos operativos alcanzaron los $4.2 millones de dólares,
representando un incremento del 12% respecto al mismo periodo del año anterior.
Sin embargo, los costos de distribución aumentaron un 18% debido al alza en los combustibles.
La junta directiva aprobó el pago de dividendos por $0.45 por acción a ser ejecutado en marzo de 2026.
"""

afirmaciones_prueba = [
    "Los ingresos del Q4 alcanzaron los $4.2 millones de dólares.",
    "Los costos de distribución disminuyeron notablemente durante el último trimestre.",
    "La empresa planea contratar 50 nuevos ingenieros en el primer semestre de 2026."
]

In [32]:
import os
from typing import List, Literal, Optional
from google.colab import userdata
from pydantic import BaseModel, Field

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


# 1. Esquema de Salida con Evidencia Textual
class VerificacionAfirmacion(BaseModel):
    afirmación_analizada: str = Field(
        description="La afirmación ingresada por el usuario"
    )
    veredicto: Literal["SOPORTADO", "CONTRADICTORIO", "NO_MENCIONADO"] = Field(
        description="Estado de veracidad de la afirmación según el documento de referencia"
    )
    cita_textual_evidencia: Optional[str] = Field(
        default=None,
        description="Extracto textual idéntico extraído del documento que respalda o refuta la afirmación. Nulo si es NO_MENCIONADO."
    )
    explicacion_auditoria: str = Field(
        description="Explicación concisa del motivo del veredicto"
    )


def crear_verificador_factico():
    """
    Crea una cadena de verificación de veracidad con salida estructurada Pydantic.
    """
    parser = PydanticOutputParser(pydantic_object=VerificacionAfirmacion)

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "Eres un verificador fáctico experto. Evalúa la 'AFIRMACIÓN' contra el 'DOCUMENTO FUENTE'.\n"
            "Tu tarea es clasificar la afirmación en SOPORTADO, CONTRADICTORIO o NO_MENCIONADO.\n"
            "Si el estado es SOPORTADO o CONTRADICTORIO, DEBES copiar la cita textual exacta del texto de origen.\n"
            "No inventes datos. Respeta estrictamente la siguiente estructura:\n{format_instructions}"
        ),
        (
            "human",
            "DOCUMENTO FUENTE:\n{documento}\n\nAFIRMACIÓN A EVALUAR:\n{afirmacion}"
        )
    ]).partial(format_instructions=parser.get_format_instructions())

    llm = ChatGoogleGenerativeAI(
        model="gemini-3.6-flash",
        temperature=0.0
    )

    return prompt | llm | parser


# =========================================
# EJECUCIÓN COMPLETA DEL EJERCICIO
# =========================================
verificador = crear_verificador_factico()

documento_auditoria = """
INFORME DE AUDITORÍA ANUAL 2025 - GRUPO FINANCIERO SUR
En el cuarto trimestre de 2025, los ingresos operativos alcanzaron los $4.2 millones de dólares,
representando un incremento del 12% respecto al mismo periodo del año anterior.
Sin embargo, los costos de distribución aumentaron un 18% debido al alza en los combustibles.
La junta directiva aprobó el pago de dividendos por $0.45 por acción a ser ejecutado en marzo de 2026.
"""

afirmaciones_prueba = [
    "Los ingresos del Q4 alcanzaron los $4.2 millones de dólares.",
    "Los costos de distribución disminuyeron notablemente durante el último trimestre.",
    "La empresa planea contratar 50 nuevos ingenieros en el primer semestre de 2026."
]

print("\n--- EJECUTANDO AUDITORÍA FÁCTICA Y MITIGACIÓN DE ALUCINACIONES ---")

for i, afirmacion in enumerate(afirmaciones_prueba, 1):
    resultado: VerificacionAfirmacion = verificador.invoke({
        "documento": documento_auditoria,
        "afirmacion": afirmacion
    })

    print(f"\n📌 CASO {i}: '{afirmacion}'")
    print(f"   • Veredicto      : {resultado.veredicto}")
    print(f"   • Cita Evidencia : \"{resultado.cita_textual_evidencia}\"")
    print(f"   • Explicación    : {resultado.explicacion_auditoria}")


--- EJECUTANDO AUDITORÍA FÁCTICA Y MITIGACIÓN DE ALUCINACIONES ---


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



📌 CASO 1: 'Los ingresos del Q4 alcanzaron los $4.2 millones de dólares.'
   • Veredicto      : SOPORTADO
   • Cita Evidencia : "En el cuarto trimestre de 2025, los ingresos operativos alcanzaron los $4.2 millones de dólares"
   • Explicación    : La afirmación es totalmente precisa y está respaldada por el documento, el cual menciona que en el cuarto trimestre (Q4) de 2025 los ingresos operativos fueron de $4.2 millones de dólares.


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



📌 CASO 2: 'Los costos de distribución disminuyeron notablemente durante el último trimestre.'
   • Veredicto      : CONTRADICTORIO
   • Cita Evidencia : "los costos de distribución aumentaron un 18% debido al alza en los combustibles."
   • Explicación    : La afirmación indica que los costos de distribución disminuyeron, pero el documento de origen declara explícitamente que aumentaron un 18%.


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



📌 CASO 3: 'La empresa planea contratar 50 nuevos ingenieros en el primer semestre de 2026.'
   • Veredicto      : NO_MENCIONADO
   • Cita Evidencia : "None"
   • Explicación    : El documento fuente no hace ninguna referencia a planes de contratación de personal o ingenieros para el año 2026.


# Evaluación Cuantitativa Tradicional: Métricas de Coincidencia, Regular Expressions y Distancia de Texto

## Determinismo y Evaluación a Costo Cero

Aunque la técnica de *LLM-as-a-Judge* (usar un modelo para evaluar a otro) es popular, resulta costosa, lenta e introduce variabilidad en la propia métrica de evaluación. En entornos de producción y auditoría, la primera línea de prueba (*testing*) para validar sistemas basados en LLM debe construirse mediante **evaluaciones cuantitativas deterministas**.

### Métricas Clave de Evaluación Tradicional:

1. **Exact Match (EM):** Métrica binaria ($0$ o $1$) que verifica si la salida del modelo es idéntica a la respuesta de referencia (*Ground Truth*). Es indispensable en la extracción de identificadores, códigos contables y fechas.
2. **Cumplimiento Estructural mediante Regex (Regex Match Rate):** Mide la proporción de respuestas que cumplen estrictamente con un patrón sintáctico o formato definido (por ejemplo, validar que una salida corresponda al patrón `^RUT-\d{8}-[0-9K]$`).
3. **Similitud Textual y Distancia de Edición (Sequence Matching / Levenshtein):** Calcula la razón de coincidencia a nivel de caracteres ($0.0$ a $1.0$) entre la respuesta generada y el valor esperado, tolerando pequeñas diferencias tipográficas.
4. **Coincidencia de Tokens / Overlap (Precision, Recall, F1-Score en N-Grams):** Evalúa qué porcentaje de las palabras clave o entidades esperadas fueron efectivamente recuperadas por el modelo.

### Práctica: Suite de Métricas Deterministas en Python

El siguiente script define funciones puras de evaluación para validar salidas de un modelo de lenguaje frente a un conjunto de prueba (*Benchmark*).


In [20]:
import re
from difflib import SequenceMatcher
from typing import Dict, List, Tuple


def evaluar_exact_match(prediccion: str, referencia: str) -> float:
    """Retorna 1.0 si la predicción limpia es idéntica a la referencia, 0.0 en caso contrario."""
    pred_limpia = prediccion.strip().lower()
    ref_limpia = referencia.strip().lower()
    return 1.0 if pred_limpia == ref_limpia else 0.0


def evaluar_patron_regex(prediccion: str, patron_regex: str) -> float:
    """Retorna 1.0 si la predicción cumple estrictamente con la expresión regular."""
    match = re.search(patron_regex, prediccion.strip())
    return 1.0 if match is not None else 0.0


def evaluar_similitud_secuencia(prediccion: str, referencia: str) -> float:
    """Calcula el ratio de similitud de Gestalt/Levenshtein entre 0.0 y 1.0."""
    return SequenceMatcher(None, prediccion.strip(), referencia.strip()).ratio()


def evaluar_f1_tokens(prediccion: str, referencia: str) -> Tuple[float, float, float]:
    """
    Calcula Precisión, Recall y F1-Score a nivel de tokens (palabras clave).
    """
    tokens_pred = set(re.findall(r'\w+', prediccion.lower()))
    tokens_ref = set(re.findall(r'\w+', referencia.lower()))

    if not tokens_pred or not tokens_ref:
        return 0.0, 0.0, 0.0

    interseccion = tokens_pred.intersection(tokens_ref)

    precision = len(interseccion) / len(tokens_pred)
    recall = len(interseccion) / len(tokens_ref)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)

    return precision, recall, f1


# --- PRUEBA RÁPIDA DE LAS MÉTRICAS ---
pred = "USD: 1500.00 | CATEGORIA: Servicios"
ref  = "USD: 1500.00 | CATEGORIA: Servicios"
patron_valido = r"^USD:\s*\d+\.\d{2}\s*\|\s*CATEGORIA:\s*\w+$"

print("--- DEMOSTRACIÓN DE MÉTRICAS DETERMINISTAS ---")
print(f"Exact Match            : {evaluar_exact_match(pred, ref)}")
print(f"Cumplimiento Regex     : {evaluar_patron_regex(pred, patron_valido)}")
print(f"Similitud de Secuencia : {evaluar_similitud_secuencia(pred, ref):.4f}")
prec, rec, f1 = evaluar_f1_tokens(pred, ref)
print(f"F1-Score Tokens        : {f1:.4f}")

--- DEMOSTRACIÓN DE MÉTRICAS DETERMINISTAS ---
Exact Match            : 1.0
Cumplimiento Regex     : 1.0
Similitud de Secuencia : 1.0000
F1-Score Tokens        : 1.0000


### Ejercicio Práctico: Benchmarking de Extracción de Códigos de Impuesto

**Objetivo:** Construir un marco de evaluación automático que ejecute un dataset de prueba contable sobre el modelo de lenguaje y genere una **Tabla Cuantitativa de Calidad** combinando Exact Match, Cumplimiento de Regex y F1-Score de Tokens.


In [34]:
import os
import re
from difflib import SequenceMatcher
from typing import Any, Dict, List
from google.colab import userdata

from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


def obtener_texto(respuesta: Any) -> str:
    """Extractor universal de texto seguro para respuestas de LangChain."""
    contenido = getattr(respuesta, "content", respuesta)
    if isinstance(contenido, str):
        return contenido.strip()
    if isinstance(contenido, list):
        partes = []
        for elem in contenido:
            if isinstance(elem, str):
                partes.append(elem)
            elif isinstance(elem, dict):
                partes.append(str(elem.get("text") or elem.get("content") or ""))
            else:
                partes.append(getattr(elem, "text", None) or str(elem))
        return "".join(partes).strip()
    return str(contenido).strip()


# 1. Dataset de Prueba (Benchmark Ground Truth)
BENCHMARK_DATASET = [
    {
        "id": 1,
        "input_factura": "Factura de retención emitida por $1,200 bajo código de impuesto IVA-RET-19.",
        "expected_code": "IVA-RET-19",
        "patron_estricto": r"^[A-Z]{3,4}-[A-Z]{3,4}-\d{2}$"
    },
    {
        "id": 2,
        "input_factura": "Se aplica la norma tributaria ISR-LOC-05 a los servicios prestados.",
        "expected_code": "ISR-LOC-05",
        "patron_estricto": r"^[A-Z]{3,4}-[A-Z]{3,4}-\d{2}$"
    },
    {
        "id": 3,
        "input_factura": "Pago exento de impuesto según resolución EXENTO-00.",
        "expected_code": "EXENTO-00",
        "patron_estricto": r"^[A-Z]{3,6}-\d{2}$"
    },
    {
        "id": 4,
        "input_factura": "Descuento aplicado bajo concepto IMP-GRV-16 sobre el subtotal.",
        "expected_code": "IMP-GRV-16",
        "patron_estricto": r"^[A-Z]{3,4}-[A-Z]{3,4}-\d{2}$"
    }
]


def crear_cadena_extractora():
    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "Extrae el código de impuesto presente en el texto.\n"
            "Responde ÚNICAMENTE con el código exacto (Ejemplo: IVA-RET-19). "
            "No agregues texto adicional, puntos ni explicaciones."
        ),
        ("human", "{input_factura}")
    ])

    llm = ChatGoogleGenerativeAI(
        model="gemini-3.6-flash",
        temperature=0.0
    )

    return prompt | llm


def ejecutar_evaluacion_benchmark(dataset: List[Dict[str, Any]]) -> Dict[str, float]:
    cadena = crear_cadena_extractora()

    total_casos = len(dataset)
    exact_matches = 0
    regex_matches = 0
    sum_similitud = 0.0

    print("\n" + "="*80)
    print(f"{'ID':<4} | {'ESPERADO':<12} | {'PREDICCIÓN LLM':<15} | {'EM':<5} | {'REGEX':<6} | {'SIMILITUD':<8}")
    print("="*80)

    for item in dataset:
        respuesta_raw = cadena.invoke({"input_factura": item["input_factura"]})
        prediccion = obtener_texto(respuesta_raw)

        # Cálculo de Métricas
        em = 1.0 if prediccion.strip() == item["expected_code"] else 0.0
        regex_ok = 1.0 if re.match(item["patron_estricto"], prediccion.strip()) else 0.0
        similitud = SequenceMatcher(None, prediccion.strip(), item["expected_code"]).ratio()

        # Acumuladores
        exact_matches += em
        regex_matches += regex_ok
        sum_similitud += similitud

        print(f"{item['id']:<4} | {item['expected_code']:<12} | {prediccion:<15} | {int(em):<5} | {int(regex_ok):<6} | {similitud:.4f}")

    print("="*80)

    # Consolidado de Calidad
    metricas_globales = {
        "exact_match_accuracy": (exact_matches / total_casos) * 100,
        "regex_compliance_rate": (regex_matches / total_casos) * 100,
        "average_similarity": (sum_similitud / total_casos) * 100
    }

    return metricas_globales


# =========================================
# EJECUCIÓN DEL BENCHMARK
# =========================================
resultados_evaluacion = ejecutar_evaluacion_benchmark(BENCHMARK_DATASET)

print("\n📊 REPORTE DE CALIDAD DEL PIPELINE (METRICAS DETERMINISTAS)")
print(f"• Exact Match Accuracy    : {resultados_evaluacion['exact_match_accuracy']:.2f}%")
print(f"• Regex Compliance Rate  : {resultados_evaluacion['regex_compliance_rate']:.2f}%")
print(f"• Average Similarity Rate : {resultados_evaluacion['average_similarity']:.2f}%")


ID   | ESPERADO     | PREDICCIÓN LLM  | EM    | REGEX  | SIMILITUD


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


1    | IVA-RET-19   | IVA-RET-19      | 1     | 1      | 1.0000


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


2    | ISR-LOC-05   | ISR-LOC-05      | 1     | 1      | 1.0000


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


3    | EXENTO-00    | EXENTO-00       | 1     | 1      | 1.0000


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


4    | IMP-GRV-16   | IMP-GRV-16      | 1     | 1      | 1.0000

📊 REPORTE DE CALIDAD DEL PIPELINE (METRICAS DETERMINISTAS)
• Exact Match Accuracy    : 100.00%
• Regex Compliance Rate  : 100.00%
• Average Similarity Rate : 100.00%


# Optimización de Eficiencia Operativa: Medición de Latencia, Rendimiento y Consumo de Tokens

## Control de Recursos, Costos y Rendimiento en Producción

En entornos corporativos y de misión crítica, una aplicación basada en LLM no solo debe ser precisa, sino también **eficiente, escalable y financieramente viable**. La optimización de eficiencia operativa abarca la supervisión de tres variables métricas fundamentales:

1. **Consumo y Distribución de Tokens:** Diferenciar estrictamente entre *Prompt Tokens* (entrada) y *Completion Tokens* (salida). Los *Completion Tokens* suelen ser considerablemente más costosos y lentos de generar debido a la naturaleza autorregresiva de los modelos.
2. **Latencia del Sistema:** Medida en milisegundos o segundos, desglosando la latencia de red, tiempo hasta el primer token (*Time to First Token* - TTFT) y tiempo de inferencia total.
3. **Rendimiento (*****Throughput*****):** Velocidad de generación medida en **Tokens por Segundo (TPS)**.

$$
\text{TPS} = \frac{\text{Tokens de salida}}{\text{Tiempo total de generación (segundos)}}
$$

### Práctica: Profiler Personalizado y Contador de Costos / Tokens

LangChain permite instrumentar llamadas usando *callbacks* nativos o envoltorios personalizados para capturar el metadato exacto del consumo y medir los tiempos de ejecución sin depender de plataformas de pago de terceros.


In [22]:
import os
import time
from typing import Any, Dict
from google.colab import userdata

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.outputs import LLMResult
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


class ProfilerOperativoHandler(BaseCallbackHandler):
    """Callback personalizado para medir latencia, tokens y calcular throughput (TPS)."""

    def __init__(self):
        self.inicio_tiempo: float = 0.0
        self.fin_tiempo: float = 0.0
        self.metricas: Dict[str, Any] = {}

    def on_llm_start(self, serialized: Dict[str, Any], prompts: list[str], **kwargs: Any) -> None:
        self.inicio_tiempo = time.perf_counter()

    def on_llm_end(self, response: LLMResult, **kwargs: Any) -> None:
        self.fin_tiempo = time.perf_counter()
        latencia_total = self.fin_tiempo - self.inicio_tiempo

        # Extracción de metadatos de uso de tokens
        llm_output = response.llm_output or {}
        token_usage = llm_output.get("token_usage", {})

        # Respaldo de extracción para diferentes integraciones
        if not token_usage and response.generations:
            gen = response.generations[0][0]
            if hasattr(gen, "message") and hasattr(gen.message, "usage_metadata"):
                usage = gen.message.usage_metadata
                token_usage = {
                    "prompt_tokens": usage.get("input_tokens", 0),
                    "completion_tokens": usage.get("output_tokens", 0),
                    "total_tokens": usage.get("total_tokens", 0)
                }

        prompt_tokens = token_usage.get("prompt_tokens", 0)
        completion_tokens = token_usage.get("completion_tokens", 0)
        total_tokens = token_usage.get("total_tokens", prompt_tokens + completion_tokens)

        tps = (completion_tokens / latencia_total) if latencia_total > 0 else 0.0

        self.metricas = {
            "latencia_segundos": round(latencia_total, 4),
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": total_tokens,
            "throughput_tps": round(tps, 2)
        }


# --- DEMOSTRACIÓN DE USO DEL PROFILER ---
profiler = ProfilerOperativoHandler()

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un analista financiero. Sé directo, conciso y técnico."),
    ("human", "Resume en 3 puntos clave la diferencia entre CAPEX y OPEX.")
])

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.0
)

# Invocación pasando el callback
cadena = prompt | llm
respuesta = cadena.invoke(
    {"input": "analisis"},
    config={"callbacks": [profiler]}
)

print("--- REPORTE DE RENDIMIENTO INDIVIDUAL ---")
print(f"• Latencia Total     : {profiler.metricas['latencia_segundos']} s")
print(f"• Prompt Tokens      : {profiler.metricas['prompt_tokens']}")
print(f"• Completion Tokens  : {profiler.metricas['completion_tokens']}")
print(f"• Total Tokens       : {profiler.metricas['total_tokens']}")
print(f"• Rendimiento (TPS)  : {profiler.metricas['throughput_tps']} tokens/seg")

--- REPORTE DE RENDIMIENTO INDIVIDUAL ---
• Latencia Total     : 39.9293 s
• Prompt Tokens      : 30
• Completion Tokens  : 1054
• Total Tokens       : 1084
• Rendimiento (TPS)  : 26.4 tokens/seg


### Ejercicio Práctico Resuelto: Benchmarking de Estrategias de Redacción (Compresión vs. Detalle)

**Objetivo:** Desarrollar un harness de pruebas comparativo que evalúe la **eficiencia operativa** (latencia, consumo de tokens y costo estimado) al ejecutar la misma tarea con dos estrategias de prompting distintas:

1. **Estrategia Extensa:** Prompt estándar sin restricción de longitud.
2. **Estrategia Optimizada:** Prompt con instrucciones de compresión estricta (*Budget-Constrained Prompting*).


In [38]:
import os
import time
from typing import Any, Dict, List
from google.colab import userdata

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.outputs import LLMResult
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


def obtener_texto(respuesta: Any) -> str:
    """Extractor universal de texto seguro para respuestas de LangChain."""
    contenido = getattr(respuesta, "content", respuesta)
    if isinstance(contenido, str):
        return contenido.strip()
    if isinstance(contenido, list):
        partes = []
        for elem in contenido:
            if isinstance(elem, str):
                partes.append(elem)
            elif isinstance(elem, dict):
                partes.append(str(elem.get("text") or elem.get("content") or ""))
            else:
                partes.append(getattr(elem, "text", None) or str(elem))
        return "".join(partes).strip()
    return str(contenido).strip()


class MonitorMetricas(BaseCallbackHandler):
    """Callback para auditoría de recursos."""
    def __init__(self):
        self.latencia = 0.0
        self.prompt_tokens = 0
        self.completion_tokens = 0
        self.total_tokens = 0

    def on_llm_start(self, serialized: Dict, prompts: List[str], **kwargs):
        self._start_time = time.perf_counter()

    def on_llm_end(self, response: LLMResult, **kwargs):
        self.latencia = time.perf_counter() - self._start_time

        # Extracción de consumo de tokens
        if response.generations:
            gen = response.generations[0][0]
            if hasattr(gen, "message") and hasattr(gen.message, "usage_metadata"):
                usage = gen.message.usage_metadata or {}
                self.prompt_tokens = usage.get("input_tokens", 0)
                self.completion_tokens = usage.get("output_tokens", 0)
                self.total_tokens = usage.get("total_tokens", 0)


def ejecutar_benchmark_eficiencia():
    llm = ChatGoogleGenerativeAI(
        model="gemini-3.5-flash-lite",
        temperature=0.0
    )

    texto_financiero = """
    El Grupo Financiero S.A. registró un incremento del 15% en sus ingresos netos durante el ejercicio fiscal 2025,
    alcanzando los $50 millones de dólares. Sin embargo, los gastos operacionales crecieron un 22% a causa de la
    expansión de infraestructura tecnológica y la contratación de personal especializado en ciberseguridad.
    La cartera de mora se mantuvo controlada en un 2.1%, mientras que el margen de liquidez cerró en 18.5%.
    """

    # 1. Prompt Extenso
    prompt_extenso = ChatPromptTemplate.from_messages([
        ("system", "Eres un asistente de negocios. Genera un informe detallado sobre el estado financiero del cliente."),
        ("human", "Analiza el siguiente texto y proporciona tus conclusiones:\n{texto}")
    ])

    # 2. Prompt Optimizado (Restricción de Tokens)
    prompt_optimizado = ChatPromptTemplate.from_messages([
        ("system", "Eres un analista de datos. Sintetiza la información en máximo 30 palabras utilizando viñetas."),
        ("human", "Analiza el siguiente texto:\n{texto}")
    ])

    casos = [
        ("Estrategia 1: Sin Restricción (Extenso)", prompt_extenso),
        ("Estrategia 2: Compresión Estricta (Optimizado)", prompt_optimizado)
    ]

    print("\n" + "="*85)
    print("📊 BENCHMARK COMPARATIVO DE EFICIENCIA OPERATIVA")
    print("="*85)

    TARIFA_PROMPT_PER_1K = 0.000075
    TARIFA_COMPLETION_PER_1K = 0.0003

    for nombre, prompt_template in casos:
        monitor = MonitorMetricas()
        cadena = prompt_template | llm

        respuesta = cadena.invoke(
            {"texto": texto_financiero},
            config={"callbacks": [monitor]}
        )

        # Extracción segura de texto
        texto_limpio = obtener_texto(respuesta)

        tps = monitor.completion_tokens / monitor.latencia if monitor.latencia > 0 else 0
        costo_estimado = (
            (monitor.prompt_tokens / 1000) * TARIFA_PROMPT_PER_1K +
            (monitor.completion_tokens / 1000) * TARIFA_COMPLETION_PER_1K
        )

        print(f"\n📌 {nombre}")
        print(f"   • Latencia Total     : {monitor.latencia:.3f} segundos")
        print(f"   • Prompt Tokens      : {monitor.prompt_tokens}")
        print(f"   • Completion Tokens  : {monitor.completion_tokens}")
        print(f"   • Total Tokens       : {monitor.total_tokens}")
        print(f"   • Rendimiento (TPS)  : {tps:.2f} tokens/s")
        print(f"   • Costo Estimado USD : ${costo_estimado:.6f}")
        print(f"   • Extracto Salida    : \"{texto_limpio[:80]}...\"")
        print("-" * 85)


# =========================================
# EJECUCIÓN DEL BENCHMARK DE RENDIMIENTO
# =========================================
ejecutar_benchmark_eficiencia()


📊 BENCHMARK COMPARATIVO DE EFICIENCIA OPERATIVA


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



📌 Estrategia 1: Sin Restricción (Extenso)
   • Latencia Total     : 4.179 segundos
   • Prompt Tokens      : 151
   • Completion Tokens  : 925
   • Total Tokens       : 1076
   • Rendimiento (TPS)  : 221.33 tokens/s
   • Costo Estimado USD : $0.000289
   • Extracto Salida    : "Como asistente de negocios, he analizado los datos financieros proporcionados co..."
-------------------------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



📌 Estrategia 2: Compresión Estricta (Optimizado)
   • Latencia Total     : 1.018 segundos
   • Prompt Tokens      : 151
   • Completion Tokens  : 50
   • Total Tokens       : 201
   • Rendimiento (TPS)  : 49.13 tokens/s
   • Costo Estimado USD : $0.000026
   • Extracto Salida    : "* Ingresos netos: $50M (+15%).
* Gastos operacionales: +22% por tecnología y per..."
-------------------------------------------------------------------------------------
